In [ ]:
# [notebook_constants]

USE_KAGGLE_SANDBOX = True
KAGGLE_SANDBOX_KWARGS = {
    'dataset_data_sources': ['greynearing/camels-us-531-era'],
    'enable_gpu': True,
    'enable_internet': False,
    'machine_shape': 'NvidiaTeslaT4Highmem',
}
C_PUCT = 2.5

# CAMELS-US ERA — Gauged Multi-Forcing Multi-Paradigm (Newman Period)

## Objective
Maximize the **Median Nash-Sutcliffe Efficiency (NSE)** across all **531 CAMELS-US basins** on the out-of-sample evaluation period (`1999-10-01` to `2003-09-30`, 4 water years, `1,461` days), strictly within the Newman et al. (2015) training window (`1998-10-01` to `2008-09-30`) with zero data leakage from `1989–1999`.

## Data & Split Protocol
- **Spatial Regime (Gauged)**: All 531 basins have training streamflow observations `QOBS_TRAIN` (`mm/day`) available during the weights training period (`2003-10-01` to `2008-09-30`, 5 water years, `1,827` days).
- **Strict Newman Temporal Containment (`1998-10-01` to `2008-09-30`, `3,653` days)**:
  - `1998-10-01` to `1999-09-30` (`365` days): 1-year meteorological spin-up prior to evaluation (`QOBS_TRAIN = NaN`).
  - `1999-10-01` to `2003-09-30` (`1,461` days, `ERA_EVAL_MASK`): Out-of-sample ERA hill-climb evaluation period (`QOBS_TRAIN = NaN`). The final 365 days (`2002-10-01` to `2003-09-30`) also serve as the 1-year meteorological spin-up prior to `2003-10-01`.
  - `2003-10-01` to `2008-09-30` (`1,827` days, `ERA_TRAIN_MASK`): ERA model weights training period (`QOBS_TRAIN` observed).
- **Inputs Available**:
  - `DYNAMIC_INPUTS`: shape `(531, 3653, 15)` (`float32`) — 5 daily meteorological variables (`prcp`, `srad`, `tmax`, `tmin`, `vp`) across 3 forcing products (`Daymet`, `Maurer`, `NLDAS`).
  - `AUX_FORCINGS`: shape `(531, 3653, 6)` (`float32`) — daily `dayl` (columns `0, 2, 4`) and `swe` (columns `1, 3, 5`, which are zero in the raw array, making explicit 3-forcing continuous degree-day snowpack accounting essential!).
  - `STATIC_ATTRIBUTES`: shape `(531, 27)` (`float32`) — 27 catchment climate, topography, soil, geology, and vegetation attributes (plus `BASIN_AREAS`, `LAT`, `LON`).
- **Required Output**:
  - The mutable cell must produce `Q_PRED` as a `float32` NumPy array of shape `(531, 3653)` containing daily streamflow predictions (`mm/day`).

## Multi-Paradigm & Physical-Neural Engineering Guidance
1. **Multi-Paradigm Architectural Diversity Across Both T4 GPUs**: Combine complementary sequence modeling paradigms in parallel across `cuda:0` and `cuda:1` — e.g., Entity-Aware 2-layer LSTMs (`EntityAwareHydroLSTM`), Multi-Scale Causal Dilated `Conv1D` + Dual `GRU-LSTM` hybrids (`MultiScaleCausalConvGRULSTM`), and Static-FiLM Dual-Pathway `GRU + LSTM` architectures (`DualPathwayFiLMGRULSTM`).
2. **Continuous 10-Year 3-Forcing Snowpack & Soil-Moisture Accounting**: Because raw `AUX_FORCINGS` SWE columns are zero, compute continuous 10-year (`1998-10-01` to `2008-09-30`) degree-day snow accumulation/melt (`SWE` and liquid water input `rainfall + snowmelt`) and nonlinear soil-moisture / antecedent precipitation index (`API`) states across all 3 forcing products (`Daymet`, `Maurer`, `NLDAS`).
3. **Avoid Unregularized Per-Basin Lookup Embeddings**: Do not use unregularized `nn.Embedding(531, D)` tables that memorize 5-year basin climate anomalies during `2003–2008`; instead, condition initial states `(h0, c0)`, input projections, and FiLM gates strictly on regularized static catchment attributes (`x_stat` with `LayerNorm + SiLU + Dropout`).
4. **Hybrid Continuous + Sliding-Window Inference**: Blend continuous 10-year rollout predictions with overlapping sliding-window inference (`365-day spin-up + 365-day target`, stride `182` days) to eliminate multi-year hidden-state drift.

In [ ]:
import os
import time
import math
import json
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

_SEARCH_DIRS = [
    Path("/kaggle/input/camels-us-531-era"),
    Path("/kaggle/input/datasets/greynearing/camels-us-531-era"),
    Path(__file__).resolve().parent.parent / "splits" if "__file__" in globals() else Path("../splits"),
    Path("../splits"),
    Path("./splits"),
]

def _find_file(filename: str) -> Path:
    for d in _SEARCH_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {filename} in any of {_SEARCH_DIRS}")

_train_npz = np.load(_find_file("camels_era_train.npz"), allow_pickle=True)
_basins_json = json.loads(_find_file("basins_531.json").read_text(encoding="utf-8"))

BASINS = _train_npz["basins"].astype(str)
_canonical_basins = [str(b) for b in _basins_json]
assert list(BASINS) == _canonical_basins, "BASINS order mismatch against canonical basins_531.json"

# Strictly enforce containment to the Newman training window (1998-10-01 to 2008-09-30, 3653 days)
_raw_dates = np.array([str(d) for d in _train_npz["dates"]])
_newman_window = (_raw_dates >= "1998-10-01") & (_raw_dates <= "2008-09-30")
DATES = _raw_dates[_newman_window]
assert len(DATES) == 3653 and DATES[0] == "1998-10-01" and DATES[-1] == "2008-09-30"

DYNAMIC_INPUTS = _train_npz["dynamic_inputs"][:, _newman_window, :].astype(np.float32)
DYNAMIC_FEATURE_NAMES = [str(x) for x in _train_npz["dynamic_feature_names"]]
AUX_FORCINGS = _train_npz["aux_forcings"][:, _newman_window, :].astype(np.float32)
AUX_FEATURE_NAMES = [str(x) for x in _train_npz["aux_feature_names"]]
STATIC_ATTRIBUTES = _train_npz["static_attributes"].astype(np.float32)
STATIC_ATTRIBUTE_NAMES = [str(x) for x in _train_npz["static_attribute_names"]]

ERA_TRAIN_MASK = (DATES >= "2003-10-01") & (DATES <= "2008-09-30")
ERA_EVAL_MASK = (DATES >= "1999-10-01") & (DATES <= "2003-09-30")
assert int(ERA_TRAIN_MASK.sum()) == 1827, f"Expected 1827 training days, got {int(ERA_TRAIN_MASK.sum())}"
assert int(ERA_EVAL_MASK.sum()) == 1461, f"Expected 1461 eval days, got {int(ERA_EVAL_MASK.sum())}"

QOBS_TRAIN = np.where(
    ERA_TRAIN_MASK[None, :],
    _train_npz["qobs_train"][:, _newman_window].astype(np.float32),
    np.nan,
).astype(np.float32)

BASIN_AREAS = _train_npz["basin_areas"].astype(np.float32)
LAT = _train_npz["lat"].astype(np.float32)
LON = _train_npz["lon"].astype(np.float32)

SPATIAL_REGIME = "gauged"
print(f"Loaded CAMELS-US 531 dataset ({SPATIAL_REGIME}): {len(BASINS)} basins, {len(DATES)} days ({DATES[0]} to {DATES[-1]}).")
print(f"ERA_TRAIN_MASK (2003-10-01..2008-09-30): {int(ERA_TRAIN_MASK.sum())} days | ERA_EVAL_MASK (1999-10-01..2003-09-30): {int(ERA_EVAL_MASK.sum())} days")
print(f"Dynamic inputs: {DYNAMIC_INPUTS.shape}, Aux: {AUX_FORCINGS.shape}, Static: {STATIC_ATTRIBUTES.shape}")
print(f"Available GPUs: {NUM_GPUS} (Primary device: {DEVICE})")

# ==============================================================================
# Precomputed Multi-Paradigm Features, Base Architectures & Inference Helpers
# ==============================================================================
import copy

def _causal_rolling(x: np.ndarray, w: int) -> np.ndarray:
    res = np.zeros_like(x, dtype=np.float32)
    cs = np.cumsum(x, axis=1, dtype=np.float64)
    res[:, w:] = ((cs[:, w:] - cs[:, :-w]) / float(w)).astype(np.float32)
    res[:, :w] = (cs[:, :w] / np.arange(1, w + 1, dtype=np.float64)[None, :]).astype(np.float32)
    return res

def _compute_3forcing_snowpack(x_dyn: np.ndarray, tc: float = 0.5, mf: float = 3.5) -> tuple[np.ndarray, np.ndarray]:
    """Continuous 10-year (1998-10-01 to 2008-09-30) degree-day snowpack & liquid water delivery across Daymet, Maurer, NLDAS."""
    n_b, n_t, _ = x_dyn.shape
    p_3f = x_dyn[:, :, [0, 5, 10]].astype(np.float32)
    t_3f = (0.5 * (x_dyn[:, :, [2, 7, 12]] + x_dyn[:, :, [3, 8, 13]])).astype(np.float32)
    swe = np.zeros((n_b, 3), dtype=np.float32)
    liq_out = np.zeros((n_b, n_t, 3), dtype=np.float32)
    swe_out = np.zeros((n_b, n_t, 3), dtype=np.float32)
    for t in range(n_t):
        pt = p_3f[:, t, :]
        tt = t_3f[:, t, :]
        snow_frac = 1.0 / (1.0 + np.exp((tt - tc) * 1.5))
        snowfall = pt * snow_frac
        rainfall = pt * (1.0 - snow_frac)
        pot_melt = np.maximum(0.0, mf * (tt - tc))
        act_melt = np.minimum(swe, pot_melt)
        swe = swe + snowfall - act_melt
        liq_out[:, t, :] = rainfall + act_melt
        swe_out[:, t, :] = np.log1p(np.maximum(0.0, swe))
    return liq_out, swe_out

def build_rich_60_features(x_dyn: np.ndarray, x_aux: np.ndarray, dates: np.ndarray) -> np.ndarray:
    """Assemble 60 multi-scale hydroclimatic, rolling water-balance, and cross-forcing agreement features."""
    n_b, _, _ = x_dyn.shape
    p_all = x_dyn[:, :, [0, 5, 10]]
    sr_all = x_dyn[:, :, [1, 6, 11]]
    tmax_all = x_dyn[:, :, [2, 7, 12]]
    tmin_all = x_dyn[:, :, [3, 8, 13]]
    vp_all = x_dyn[:, :, [4, 9, 14]]
    tmean_all = 0.5 * (tmax_all + tmin_all)
    tdiurnal_all = tmax_all - tmin_all

    dayl_all = x_aux[:, :, [0, 2, 4]]
    swe_all = x_aux[:, :, [1, 3, 5]]

    p_mean = p_all.mean(axis=-1)
    p_std = p_all.std(axis=-1)
    t_mean = tmean_all.mean(axis=-1)
    t_std = tmean_all.std(axis=-1)
    sr_mean = sr_all.mean(axis=-1)
    vp_mean = vp_all.mean(axis=-1)
    swe_mean = swe_all.mean(axis=-1)
    dayl_mean = dayl_all.mean(axis=-1)

    pet_proxy = np.maximum(0.0, sr_mean * np.maximum(0.0, t_mean + 5.0) * (dayl_mean / 43200.0) * 0.002)
    wb_daily = p_mean - pet_proxy

    feats = [
        x_dyn,
        x_aux,
        tdiurnal_all,
        p_mean[..., None],
        p_std[..., None],
        t_mean[..., None],
        t_std[..., None],
        sr_mean[..., None],
        vp_mean[..., None],
        swe_mean[..., None],
        pet_proxy[..., None],
        np.log1p(np.maximum(0.0, p_mean))[..., None],
        np.log1p(np.maximum(0.0, swe_mean))[..., None],
    ]
    for w in (3, 7, 15, 30, 60, 120, 365):
        feats.append(_causal_rolling(p_mean, w)[..., None])
    for w in (7, 30, 90):
        feats.append(_causal_rolling(t_mean, w)[..., None])
        feats.append(_causal_rolling(wb_daily, w)[..., None])
    for k in range(3):
        pk = p_all[:, :, k]
        for w in (7, 30, 90):
            feats.append(_causal_rolling(pk, w)[..., None])

    dates_dt = pd.to_datetime(dates)
    doy = dates_dt.dayofyear.values.astype(np.float32)
    for mult in (1.0, 2.0):
        s = np.sin(2.0 * np.pi * mult * doy / 365.25).astype(np.float32)
        c = np.cos(2.0 * np.pi * mult * doy / 365.25).astype(np.float32)
        feats.append(np.tile(s, (n_b, 1))[..., None])
        feats.append(np.tile(c, (n_b, 1))[..., None])
    return np.concatenate(feats, axis=-1).astype(np.float32)

def build_clean_36_features(x_dyn: np.ndarray, x_aux: np.ndarray, dates: np.ndarray) -> np.ndarray:
    """Assemble 30 canonical features + 6 continuous 3-forcing degree-day snowpack/liquid-water features = 36 features."""
    n_basins, _, n_dyn = x_dyn.shape
    n_prods = max(1, n_dyn // 5)
    p_idx = [5 * k for k in range(n_prods)]
    sr_idx = [5 * k + 1 for k in range(n_prods)]
    t_idx = [idx for k in range(n_prods) for idx in (5 * k + 2, 5 * k + 3)]
    vp_idx = [5 * k + 4 for k in range(n_prods)]

    dates_dt = pd.to_datetime(dates)
    d_sin = np.sin(2.0 * np.pi * dates_dt.dayofyear.values / 365.25).astype(np.float32)
    d_cos = np.cos(2.0 * np.pi * dates_dt.dayofyear.values / 365.25).astype(np.float32)
    doy_feats = np.stack([np.tile(d_sin, (n_basins, 1)), np.tile(d_cos, (n_basins, 1))], axis=-1)

    p_mean = x_dyn[..., p_idx].mean(axis=-1)
    p_7d = _causal_rolling(p_mean, 7)
    p_30d = _causal_rolling(p_mean, 30)
    liq_3f, log_swe_3f = _compute_3forcing_snowpack(x_dyn)

    return np.concatenate(
        [
            x_dyn,
            x_aux,
            doy_feats,
            np.expand_dims(p_mean, -1),
            np.expand_dims(p_7d, -1),
            np.expand_dims(p_30d, -1),
            x_dyn[..., t_idx].mean(axis=-1, keepdims=True),
            x_dyn[..., sr_idx].mean(axis=-1, keepdims=True),
            x_dyn[..., vp_idx].mean(axis=-1, keepdims=True),
            np.log1p(np.maximum(p_mean[..., None], 0.0)),
            liq_3f,
            log_swe_3f,
        ],
        axis=-1,
    ).astype(np.float32)


# ==============================================================================
# Paradigm A Architectures (60 Rich Features + Basin-Conditioned + EMA)
# ==============================================================================
class CausalConvGRULSTM(nn.Module):
    def __init__(self, dyn_dim: int, stat_dim: int = 27, num_basins: int = 531, hidden_size: int = 288, dropout: float = 0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.basin_emb = nn.Embedding(num_basins, 32)
        self.stat_net = nn.Sequential(
            nn.Linear(stat_dim + 32, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
        )
        self.conv_d1 = nn.Conv1d(dyn_dim, 64, kernel_size=3, dilation=1)
        self.conv_d3 = nn.Conv1d(dyn_dim, 64, kernel_size=3, dilation=3)
        self.conv_d9 = nn.Conv1d(dyn_dim, 64, kernel_size=3, dilation=9)

        in_rec = dyn_dim + 192 + hidden_size
        self.gru = nn.GRU(in_rec, hidden_size, num_layers=1, batch_first=True)
        self.lstm = nn.LSTM(hidden_size + hidden_size, hidden_size, num_layers=1, batch_first=True)
        for name, param in self.lstm.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "bias" in name:
                nn.init.constant_(param, 0.0)
                n = param.size(0)
                param.data[n // 4 : n // 2].fill_(3.0)

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.SiLU(),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x_d: torch.Tensor, x_s: torch.Tensor, b_idx: torch.Tensor) -> torch.Tensor:
        B, T, _ = x_d.shape
        s_emb = self.stat_net(torch.cat([x_s, self.basin_emb(b_idx)], dim=-1))
        xt = x_d.transpose(1, 2)
        c1 = F.silu(self.conv_d1(F.pad(xt, (2, 0)))).transpose(1, 2)
        c3 = F.silu(self.conv_d3(F.pad(xt, (6, 0)))).transpose(1, 2)
        c9 = F.silu(self.conv_d9(F.pad(xt, (18, 0)))).transpose(1, 2)
        s_seq = s_emb.unsqueeze(1).expand(-1, T, -1)
        g_out, _ = self.gru(torch.cat([x_d, c1, c3, c9, s_seq], dim=-1))
        l_out, _ = self.lstm(torch.cat([g_out, s_seq], dim=-1))
        return self.head(g_out + l_out).squeeze(-1)


class MultiScaleFiLMEALSTM(nn.Module):
    def __init__(self, dyn_dim: int, stat_dim: int = 27, num_basins: int = 531, hidden_size: int = 320, dropout: float = 0.35):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = 2
        self.basin_emb = nn.Embedding(num_basins, 32)
        in_stat = stat_dim + 32

        self.stat_net = nn.Sequential(
            nn.Linear(in_stat, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
        )
        self.init_h = nn.Linear(hidden_size, self.num_layers * hidden_size)
        self.init_c = nn.Linear(hidden_size, self.num_layers * hidden_size)
        self.in_film = nn.Linear(hidden_size, dyn_dim * 2)
        self.out_film = nn.Linear(hidden_size, hidden_size * 2)

        self.lstm = nn.LSTM(
            dyn_dim + hidden_size,
            hidden_size,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=dropout,
        )
        for name, param in self.lstm.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.constant_(param, 0.0)
                n = param.size(0)
                param.data[n // 4 : n // 2].fill_(3.5)

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.SiLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x_d: torch.Tensor, x_s: torch.Tensor, b_idx: torch.Tensor) -> torch.Tensor:
        B, T, _ = x_d.shape
        s_emb = self.stat_net(torch.cat([x_s, self.basin_emb(b_idx)], dim=-1))
        gamma_in, beta_in = self.in_film(s_emb).chunk(2, dim=-1)
        x_d_mod = x_d * (1.0 + 0.2 * torch.tanh(gamma_in.unsqueeze(1))) + 0.2 * beta_in.unsqueeze(1)

        h0 = torch.tanh(self.init_h(s_emb)).view(self.num_layers, B, self.hidden_size).contiguous()
        c0 = torch.tanh(self.init_c(s_emb)).view(self.num_layers, B, self.hidden_size).contiguous()

        s_seq = s_emb.unsqueeze(1).expand(-1, T, -1)
        out, _ = self.lstm(torch.cat([x_d_mod, s_seq], dim=-1), (h0, c0))

        gamma_out, beta_out = self.out_film(s_emb).chunk(2, dim=-1)
        out = out * (1.0 + 0.2 * torch.tanh(gamma_out.unsqueeze(1))) + 0.2 * beta_out.unsqueeze(1)
        return self.head(out).squeeze(-1)


# ==============================================================================
# Paradigm B Architectures (36 Clean + 3-Forcing Snowpack Features, Pure Static)
# ==============================================================================
class EntityAwareHydroLSTM(nn.Module):
    def __init__(self, dyn_dim: int, stat_dim: int = 27, hidden_size: int = 320, num_layers: int = 2, dropout: float = 0.4):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.stat_embed = nn.Sequential(
            nn.Linear(stat_dim, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.Dropout(dropout),
        )
        self.init_h = nn.Linear(hidden_size, num_layers * hidden_size)
        self.init_c = nn.Linear(hidden_size, num_layers * hidden_size)
        self.lstm = nn.LSTM(
            dyn_dim + hidden_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, 1),
        )
        for name, param in self.lstm.named_parameters():
            if "weight" in name:
                nn.init.orthogonal_(param)
            elif "bias" in name:
                nn.init.constant_(param, 0.0)
                n = param.size(0)
                param.data[n // 4 : n // 2].fill_(4.0)

    def forward(self, x_dyn: torch.Tensor, x_stat: torch.Tensor) -> torch.Tensor:
        B, T, _ = x_dyn.shape
        s_emb = self.stat_embed(x_stat)
        h0 = self.init_h(s_emb).view(self.num_layers, B, self.hidden_size).contiguous()
        c0 = self.init_c(s_emb).view(self.num_layers, B, self.hidden_size).contiguous()
        s_seq = s_emb.unsqueeze(1).expand(-1, T, -1)
        out, _ = self.lstm(torch.cat([x_dyn, s_seq], dim=-1), (h0, c0))
        return self.head(out).squeeze(-1)


class DualPathwayFiLMGRULSTM(nn.Module):
    def __init__(self, dyn_dim: int, stat_dim: int = 27, hidden_size: int = 320, dropout: float = 0.4):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = 2
        self.gru_hidden = 256
        self.stat_embed = nn.Sequential(
            nn.Linear(stat_dim, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.Dropout(dropout),
        )
        self.in_film = nn.Linear(hidden_size, dyn_dim * 2)
        self.init_h = nn.Linear(hidden_size, self.num_layers * hidden_size)
        self.init_c = nn.Linear(hidden_size, self.num_layers * hidden_size)
        self.init_gru = nn.Linear(hidden_size, self.num_layers * self.gru_hidden)

        self.lstm = nn.LSTM(
            dyn_dim + hidden_size,
            hidden_size,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.gru = nn.GRU(
            dyn_dim + hidden_size,
            self.gru_hidden,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.gru_proj = nn.Linear(self.gru_hidden, hidden_size)
        self.out_film = nn.Linear(hidden_size, hidden_size * 2)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, 1),
        )
        for name, param in self.lstm.named_parameters():
            if "weight" in name:
                nn.init.orthogonal_(param)
            elif "bias" in name:
                nn.init.constant_(param, 0.0)
                n = param.size(0)
                param.data[n // 4 : n // 2].fill_(4.0)
        for name, param in self.gru.named_parameters():
            if "weight" in name:
                nn.init.orthogonal_(param)
            elif "bias" in name:
                nn.init.constant_(param, 0.0)

    def forward(self, x_dyn: torch.Tensor, x_stat: torch.Tensor) -> torch.Tensor:
        B, T, _ = x_dyn.shape
        s_emb = self.stat_embed(x_stat)
        gamma_in, beta_in = self.in_film(s_emb).chunk(2, dim=-1)
        x_mod = x_dyn * (1.0 + 0.15 * torch.tanh(gamma_in.unsqueeze(1))) + 0.15 * beta_in.unsqueeze(1)

        h0 = self.init_h(s_emb).view(self.num_layers, B, self.hidden_size).contiguous()
        c0 = self.init_c(s_emb).view(self.num_layers, B, self.hidden_size).contiguous()
        g0 = self.init_gru(s_emb).view(self.num_layers, B, self.gru_hidden).contiguous()

        s_seq = s_emb.unsqueeze(1).expand(-1, T, -1)
        inp = torch.cat([x_mod, s_seq], dim=-1)
        l_out, _ = self.lstm(inp, (h0, c0))
        g_out, _ = self.gru(inp, g0)
        out = l_out + 0.4 * self.gru_proj(g_out)

        gamma_out, beta_out = self.out_film(s_emb).chunk(2, dim=-1)
        out = out * (1.0 + 0.15 * torch.tanh(gamma_out.unsqueeze(1))) + 0.15 * beta_out.unsqueeze(1)
        return self.head(out).squeeze(-1)


t0_all = time.time()
X_rich_raw = build_rich_60_features(DYNAMIC_INPUTS, AUX_FORCINGS, DATES)
X_clean_raw = build_clean_36_features(DYNAMIC_INPUTS, AUX_FORCINGS, DATES)

stat_order = np.argsort(np.asarray(STATIC_ATTRIBUTE_NAMES, dtype=str))
X_stat_raw = np.nan_to_num(STATIC_ATTRIBUTES, nan=0.0).astype(np.float32)
X_stat_sorted_raw = np.nan_to_num(STATIC_ATTRIBUTES[:, stat_order], nan=0.0).astype(np.float32)

# Strictly compute normalization statistics on ERA_TRAIN_MASK (2003-10-01 to 2008-09-30)
r_mean = X_rich_raw[:, ERA_TRAIN_MASK, :].mean(axis=(0, 1), keepdims=True)
r_std = X_rich_raw[:, ERA_TRAIN_MASK, :].std(axis=(0, 1), keepdims=True) + 1e-6
X_rich_norm = ((X_rich_raw - r_mean) / r_std).astype(np.float32)

c_mean = X_clean_raw[:, ERA_TRAIN_MASK, :].mean(axis=(0, 1), keepdims=True)
c_std = X_clean_raw[:, ERA_TRAIN_MASK, :].std(axis=(0, 1), keepdims=True) + 1e-6
X_clean_norm = ((X_clean_raw - c_mean) / c_std).astype(np.float32)

s_mean = X_stat_raw.mean(axis=0, keepdims=True)
s_std = X_stat_raw.std(axis=0, keepdims=True) + 1e-6
X_stat_norm = ((X_stat_raw - s_mean) / s_std).astype(np.float32)

ss_mean = X_stat_sorted_raw.mean(axis=0, keepdims=True)
ss_std = X_stat_sorted_raw.std(axis=0, keepdims=True) + 1e-6
X_stat_sorted_norm = ((X_stat_sorted_raw - ss_mean) / ss_std).astype(np.float32)

valid_q = QOBS_TRAIN[:, ERA_TRAIN_MASK]
y_mean = float(np.nanmean(valid_q))
y_std = float(np.nanstd(valid_q)) + 1e-6
Y_norm = ((QOBS_TRAIN - y_mean) / y_std).astype(np.float32)

n_basins, n_total_days, _ = X_rich_norm.shape
per_basin_vars = np.ones(n_basins, dtype=np.float32)
for b in range(n_basins):
    vb = valid_q[b, ~np.isnan(valid_q[b])]
    if len(vb) > 10:
        per_basin_vars[b] = float(np.var(vb / y_std))

SEQ_LEN = 365
BATCH_SIZE = 128
SAMPLES_PER_BASIN = 25

valid_starts_a = np.asarray(
    [t0 for t0 in range(0, n_total_days - SEQ_LEN + 1) if np.sum(ERA_TRAIN_MASK[t0 + 90 : t0 + SEQ_LEN]) >= 60],
    dtype=np.int64,
)
train_idx = np.where(ERA_TRAIN_MASK)[0]
t_min_b = max(0, int(train_idx[0]) - SEQ_LEN + 32)
t_max_b = max(t_min_b, int(train_idx[-1]) - SEQ_LEN)


def _infer_cont_and_win(eval_fn, n_basins: int, n_total_days: int) -> np.ndarray:
    pred_cont = np.zeros((n_basins, n_total_days), dtype=np.float32)
    for b0 in range(0, n_basins, 64):
        b1 = min(b0 + 64, n_basins)
        pred_cont[b0:b1] = eval_fn(b0, b1, 0, n_total_days)

    pred_win_sum = np.zeros((n_basins, n_total_days), dtype=np.float64)
    pred_win_cnt = np.zeros((n_basins, n_total_days), dtype=np.float64)
    win_len = 730
    spin_len = 365
    starts = list(range(0, n_total_days - win_len + 1, 182))
    if starts[-1] != n_total_days - win_len:
        starts.append(n_total_days - win_len)
    for t0 in starts:
        t1 = t0 + win_len
        for b0 in range(0, n_basins, 64):
            b1 = min(b0 + 64, n_basins)
            ymm = eval_fn(b0, b1, t0, t1)
            if t0 == 0:
                pred_win_sum[b0:b1, :t1] += ymm
                pred_win_cnt[b0:b1, :t1] += 1.0
            else:
                pred_win_sum[b0:b1, t0 + spin_len : t1] += ymm[:, spin_len:]
                pred_win_cnt[b0:b1, t0 + spin_len : t1] += 1.0
    pred_win = (pred_win_sum / np.maximum(pred_win_cnt, 1.0)).astype(np.float32)
    return (0.5 * pred_cont + 0.5 * pred_win).astype(np.float32)


def _train_member(task: tuple[str, int, int]) -> tuple[str, np.ndarray]:
    family, seed, gpu_id = task
    dev = torch.device(f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu")
    use_amp = dev.type == "cuda"

    rng = np.random.default_rng(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    y_gpu = torch.from_numpy(Y_norm).to(dev)
    b_var_gpu = torch.from_numpy(per_basin_vars).to(dev)
    train_mask_gpu = torch.from_numpy(ERA_TRAIN_MASK).to(dev)
    seq_offsets = torch.arange(SEQ_LEN, device=dev).unsqueeze(0)
    steps_per_epoch = math.ceil((n_basins * SAMPLES_PER_BASIN) / BATCH_SIZE)

    if family in ("conv_gru_lstm", "film_ealstm"):
        x_d_gpu = torch.from_numpy(X_rich_norm).to(dev)
        x_s_gpu = torch.from_numpy(X_stat_norm).to(dev)
        epochs = 45
        if family == "conv_gru_lstm":
            model = CausalConvGRULSTM(dyn_dim=X_rich_norm.shape[-1], stat_dim=X_stat_norm.shape[-1], num_basins=n_basins).to(dev)
        else:
            model = MultiScaleFiLMEALSTM(dyn_dim=X_rich_norm.shape[-1], stat_dim=X_stat_norm.shape[-1], num_basins=n_basins).to(dev)

        ema_model = copy.deepcopy(model)
        ema_decay = 0.992
        ema_start_epoch = int(epochs * 0.45)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1.2e-3, weight_decay=5e-2)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=1.2e-3, steps_per_epoch=steps_per_epoch, epochs=epochs
        )
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

        for ep in range(1, epochs + 1):
            model.train()
            p_b = np.repeat(np.arange(n_basins), SAMPLES_PER_BASIN)
            p_t = rng.choice(valid_starts_a, size=len(p_b), replace=True)
            perm = rng.permutation(len(p_b))
            p_b, p_t = p_b[perm], p_t[perm]

            for i in range(0, len(p_b), BATCH_SIZE):
                idx_b = torch.from_numpy(p_b[i : i + BATCH_SIZE]).long().to(dev)
                idx_t = torch.from_numpy(p_t[i : i + BATCH_SIZE]).long().to(dev)
                win_t = idx_t.unsqueeze(1) + seq_offsets

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    y_hat = model(x_d_gpu[idx_b.unsqueeze(1), win_t], x_s_gpu[idx_b], idx_b)
                    y_true = y_gpu[idx_b.unsqueeze(1), win_t]
                    m = train_mask_gpu[win_t] & (~torch.isnan(y_true))
                    m[:, :90] = False
                    if bool(m.any()):
                        diff = torch.where(m, y_hat - y_true, torch.zeros_like(y_hat))
                        sq_err = diff ** 2
                        denom = (b_var_gpu[idx_b] + 0.05).unsqueeze(1)
                        norm_sq_err = sq_err / denom
                        cnt = m.float().sum(dim=1).clamp(min=1.0)
                        valid_row = m.any(dim=1)
                        traj_nse_loss = (norm_sq_err.sum(dim=1) / cnt)[valid_row]
                        delta = 2.0
                        huber_nse = (delta ** 2) * (torch.sqrt(1.0 + (traj_nse_loss / delta) ** 2) - 1.0)
                        nse_term = torch.mean(huber_nse)

                        y_true_mm = torch.clamp(y_true[m] * y_std + y_mean, min=0.0)
                        y_hat_mm = torch.clamp(y_hat[m] * y_std + y_mean, min=0.0)
                        log_term = torch.mean((torch.log1p(y_hat_mm) - torch.log1p(y_true_mm)) ** 2)
                        loss = nse_term + 0.12 * log_term

                        scaler.scale(loss).backward()
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                scheduler.step()

                if ep >= ema_start_epoch:
                    with torch.no_grad():
                        for p_ema, p_cur in zip(ema_model.parameters(), model.parameters()):
                            p_ema.data.mul_(ema_decay).add_(p_cur.data, alpha=1.0 - ema_decay)
                else:
                    with torch.no_grad():
                        for p_ema, p_cur in zip(ema_model.parameters(), model.parameters()):
                            p_ema.data.copy_(p_cur.data)

        ema_model.eval()
        all_b = torch.arange(n_basins, dtype=torch.long, device=dev)
        def _eval_a(b0: int, b1: int, t0: int, t1: int) -> np.ndarray:
            with torch.no_grad(), torch.amp.autocast("cuda", enabled=use_amp):
                yh = ema_model(x_d_gpu[b0:b1, t0:t1], x_s_gpu[b0:b1], all_b[b0:b1])
                return torch.clamp(yh * y_std + y_mean, min=0.0).float().cpu().numpy()
        return family, _infer_cont_and_win(_eval_a, n_basins, n_total_days)

    else:
        x_d_gpu = torch.from_numpy(X_clean_norm).to(dev)
        x_s_gpu = torch.from_numpy(X_stat_sorted_norm).to(dev)
        epochs = 60
        if family == "snow_film_grulstm":
            model = DualPathwayFiLMGRULSTM(dyn_dim=X_clean_norm.shape[-1], stat_dim=X_stat_sorted_norm.shape[-1]).to(dev)
        else:
            model = EntityAwareHydroLSTM(dyn_dim=X_clean_norm.shape[-1], stat_dim=X_stat_sorted_norm.shape[-1]).to(dev)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1.2e-3, weight_decay=0.1)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=1.2e-3, steps_per_epoch=steps_per_epoch, epochs=epochs
        )
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

        for _ep in range(epochs):
            model.train()
            p_b = np.repeat(np.arange(n_basins), SAMPLES_PER_BASIN)
            p_t = rng.integers(t_min_b, t_max_b + 1, size=len(p_b))
            perm = rng.permutation(len(p_b))
            p_b, p_t = p_b[perm], p_t[perm]

            for i in range(0, len(p_b), BATCH_SIZE):
                idx_b = torch.from_numpy(p_b[i : i + BATCH_SIZE]).long().to(dev)
                idx_t = torch.from_numpy(p_t[i : i + BATCH_SIZE]).long().to(dev)
                win_t = idx_t.unsqueeze(1) + seq_offsets

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    y_hat = model(x_d_gpu[idx_b.unsqueeze(1), win_t], x_s_gpu[idx_b])
                    y_true = y_gpu[idx_b.unsqueeze(1), win_t]
                    m = train_mask_gpu[win_t] & (~torch.isnan(y_true))
                    if bool(m.any()):
                        nse_loss = torch.mean(
                            ((y_hat[m] - y_true[m]) ** 2)
                            / (b_var_gpu[idx_b].unsqueeze(1).expand(-1, SEQ_LEN)[m] + 0.05)
                        )
                        y_true_unnorm = torch.clamp(y_true[m] * y_std + y_mean, min=0.0)
                        y_hat_unnorm = torch.clamp(y_hat[m] * y_std + y_mean, min=0.0)
                        log_loss = torch.mean((torch.log1p(y_hat_unnorm) - torch.log1p(y_true_unnorm)) ** 2)
                        loss = nse_loss + 0.10 * log_loss

                        scaler.scale(loss).backward()
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                scheduler.step()

        model.eval()
        def _eval_b(b0: int, b1: int, t0: int, t1: int) -> np.ndarray:
            with torch.no_grad(), torch.amp.autocast("cuda", enabled=use_amp):
                yh = model(x_d_gpu[b0:b1, t0:t1], x_s_gpu[b0:b1])
                return torch.clamp(yh * y_std + y_mean, min=0.0).float().cpu().numpy()
        return family, _infer_cont_and_win(_eval_b, n_basins, n_total_days)



Loaded CAMELS-US 531 dataset (gauged): 531 basins, 3653 days (1998-10-01 to 2008-09-30).
ERA_TRAIN_MASK (2003-10-01..2008-09-30): 1827 days | ERA_EVAL_MASK (1999-10-01..2003-09-30): 1461 days
Dynamic inputs: (531, 3653, 15), Aux: (531, 3653, 6), Static: (531, 27)
Available GPUs: 2 (Primary device: cuda)


In [ ]:
# [begin_stage]
# Begin Stage

In [ ]:
# [stage_constants]
# [exclude_from_prompt]

INSTRUCTION = [
    "Keep the 7 base tasks via _train_member(task) from Cell 2, and add a 5th complementary model family (e.g. a Dilated WaveNet/TCN + LSTM hybrid or Cross-Forcing Attention GRU trained on X_rich_norm/X_stat_norm or X_clean_norm/X_stat_sorted_norm using _infer_cont_and_win), then blend its predictions with pred_conv, pred_film, pred_snow_film, and pred_snow_ea.",
    "Learn a static-attribute-conditioned or climate-regime-gated per-basin blending weight network (using X_stat_norm or X_stat_sorted_norm, e.g. aridity/snow-fraction/baseflow-index gating or out-of-fold validation on the first year of ERA_TRAIN_MASK) to combine pred_conv, pred_film, pred_snow_film, and pred_snow_ea per basin instead of uniform global weights.",
    "Add additional seeds and architectural diversity (such as a Multi-Scale Bidirectional/Causal State-Space or Residual Highway GRU-LSTM with EMA weight averaging and multi-horizon routing) trained on GPU 0 and GPU 1 in parallel via ThreadPoolExecutor, and evaluate with _infer_cont_and_win.",
    "Explore non-linear / rank-aware / trimmed / power-transformed ensemble aggregation across the 4+ model families (e.g. combining arithmetic mean on low/medium flows with geometric/power mean or variance-matched scaling on peak flows, conditioned on basin snow fraction and aridity).",
    "Implement a custom training task in the rewrite cell that augments X_rich_norm or X_clean_norm with learned cross-forcing disagreement attention (weighting Daymet vs Maurer vs NLDAS dynamically per basin and season) and blends its _infer_cont_and_win predictions into Q_PRED.",
    "Synthesize the strongest new model family, multi-seed expansion, and static-attribute-conditioned blending rules discovered so far on top of the 4 base families (pred_conv, pred_film, pred_snow_film, pred_snow_ea) to maximize 531-basin Median NSE on ERA_VAL_MASK."
]
VALUE_REGEX = r"(?m)^Candidate metric to be maximized: (.*)$"
OBJECTIVE = "maximize"
WORST_VALUE = -2e308


In [ ]:
# [rewrite_cell]
# ==============================================================================
# Multi-Paradigm Dual-GPU Ensemble & Architecture Search (Node 0 = 0.8410 Median NSE)
# Pre-loaded in global scope from Cell 2:
#   - X_rich_norm (531, 3653, 60), X_stat_norm (531, 79)
#   - X_clean_norm (531, 3653, 36), X_stat_sorted_norm (531, 27)
#   - Y_norm (531, 3653), y_mean, y_std, per_basin_vars (531,), ERA_TRAIN_MASK (3653,)
#   - valid_starts_a, t_min_b, t_max_b, n_basins, n_total_days, SEQ_LEN, BATCH_SIZE, SAMPLES_PER_BASIN
#   - Base architectures: CausalConvGRULSTM, MultiScaleFiLMEALSTM, DualPathwayFiLMGRULSTM, EntityAwareHydroLSTM
#   - Helpers: _infer_cont_and_win(eval_fn, n_basins, n_total_days) and _train_member((family, seed, gpu_id))
# ==============================================================================

n_gpus = max(1, NUM_GPUS)
base_tasks = [
    ("conv_gru_lstm", 42, 0 % n_gpus),
    ("conv_gru_lstm", 101, 0 % n_gpus),
    ("conv_gru_lstm", 2024, 0 % n_gpus),
    ("film_ealstm", 42, 0 % n_gpus),
    ("snow_film_grulstm", 42, 1 % n_gpus),
    ("snow_film_grulstm", 101, 1 % n_gpus),
    ("snow_ealstm", 42, 1 % n_gpus),
]

with ThreadPoolExecutor(max_workers=min(2, n_gpus)) as ex:
    results = list(ex.map(_train_member, base_tasks))

by_fam: dict[str, list[np.ndarray]] = {}
for fam, pred in results:
    by_fam.setdefault(fam, []).append(pred)

pred_conv = np.mean(by_fam["conv_gru_lstm"], axis=0)
pred_film = np.mean(by_fam["film_ealstm"], axis=0)
pred_snow_film = np.mean(by_fam["snow_film_grulstm"], axis=0)
pred_snow_ea = np.mean(by_fam["snow_ealstm"], axis=0)

Q_PRED = (
    0.38 * pred_conv
    + 0.12 * pred_film
    + 0.36 * pred_snow_film
    + 0.14 * pred_snow_ea
).astype(np.float32)
print(f"Completed 7-member 4-family Multi-Paradigm Gauged Ensemble across {n_gpus} GPU(s) in {time.time() - t0_all:.1f}s.")


In [ ]:
# ==============================================================================
# Immutable ERA Hill-Climb Evaluation (Strictly 1999-10-01 to 2003-09-30)
# ==============================================================================
_eval_npz = np.load(_find_file("era_eval_targets.npz"), allow_pickle=True)
_eval_raw_dates = np.array([str(d) for d in _eval_npz["dates"]])
_eval_window = (_eval_raw_dates >= "1998-10-01") & (_eval_raw_dates <= "2008-09-30")
_qobs_eval = _eval_npz["qobs_eval"][:, _eval_window].astype(np.float32)

assert "Q_PRED" in globals(), "Mutable cell must define Q_PRED!"
Q_PRED = np.asarray(Q_PRED, dtype=np.float32)
assert Q_PRED.shape == (len(BASINS), len(DATES)), (
    f"Expected Q_PRED shape {(len(BASINS), len(DATES))}, got {Q_PRED.shape}"
)
assert not np.any(np.isnan(Q_PRED[:, ERA_EVAL_MASK])) and not np.any(np.isinf(Q_PRED[:, ERA_EVAL_MASK])), (
    "Q_PRED contains NaN or Inf on ERA_EVAL_MASK (1999-10-01 to 2003-09-30)!"
)

def _compute_basin_nses(q_pred: np.ndarray, q_true: np.ndarray, mask: np.ndarray) -> np.ndarray:
    nses = []
    for i in range(len(BASINS)):
        yt = q_true[i, mask]
        yp = q_pred[i, mask]
        valid = ~np.isnan(yt)
        if np.sum(valid) < 30:
            continue
        yt_v = yt[valid]
        yp_v = yp[valid]
        denom = np.sum((yt_v - np.mean(yt_v)) ** 2)
        if denom > 1e-12:
            nses.append(1.0 - float(np.sum((yp_v - yt_v) ** 2) / denom))
    return np.asarray(nses, dtype=np.float64)

_nses_eval = _compute_basin_nses(Q_PRED, _qobs_eval, ERA_EVAL_MASK)
_median_nse = float(np.median(_nses_eval))
_mean_nse = float(np.mean(_nses_eval))
_q25, _q75 = float(np.percentile(_nses_eval, 25)), float(np.percentile(_nses_eval, 75))
_neg_count = int(np.sum(_nses_eval < 0.0))

print(f"=== ERA Evaluation Summary ({SPATIAL_REGIME} | {len(_nses_eval)} Basins | 1999-10-01 to 2003-09-30) ===")
print(f"Median NSE (1999-2003): {_median_nse:.5f} | Mean NSE: {_mean_nse:.5f} | IQR: [{_q25:.5f}, {_q75:.5f}] | Basins < 0: {_neg_count}")
print(f"Candidate metric to be maximized: {_median_nse:.6f}")

=== ERA Evaluation Summary (gauged | 531 Basins | 1999-10-01 to 2003-09-30) ===
Median NSE (1999-2003): 0.84100 | Mean NSE: 0.78120 | IQR: [0.74200, 0.88400] | Basins < 0: 3
Candidate metric to be maximized: 0.841000


In [ ]:
# [end_stage]
# End Stage